In [1]:
"""
explore.py
----------
A simple script to explore and understand the raw_scrape table.
Run this anytime you want to "see" what's in the database.

Usage:
    python explore.py
"""

import duckdb
import pandas as pd

# Make pandas show full content without truncating
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

con = duckdb.connect("scrape.duckdb")

def divider(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

# ─────────────────────────────────────────────
# 1. BASIC SHAPE — how big is the data?
# ─────────────────────────────────────────────
divider("1. BASIC SHAPE")

total_rows = con.execute("SELECT COUNT(*) FROM raw_scrape").fetchone()[0]
total_cols = len(con.execute("SELECT * FROM raw_scrape LIMIT 1").df().columns)
print(f"  Total rows    : {total_rows:,}")
print(f"  Total columns : {total_cols}")



  1. BASIC SHAPE
  Total rows    : 245,169
  Total columns : 13


In [2]:

# ─────────────────────────────────────────────
# 2. COLUMN NAMES AND TYPES
# ─────────────────────────────────────────────
divider("2. COLUMNS AND DATA TYPES")

schema = con.execute("DESCRIBE raw_scrape").df()
print(schema[["column_name", "column_type"]].to_string(index=False))



  2. COLUMNS AND DATA TYPES
   column_name column_type
   scrape_date     VARCHAR
      category     VARCHAR
  company_name     VARCHAR
         phone     VARCHAR
         email     VARCHAR
       website     VARCHAR
employee_count     VARCHAR
          city     VARCHAR
       country     VARCHAR
    scraped_at     VARCHAR
  _source_file     VARCHAR
    _file_date     VARCHAR
    source_url     VARCHAR


In [13]:

# ─────────────────────────────────────────────
# 3. FIRST 5 ROWS — what does real data look like?
# ─────────────────────────────────────────────
divider("3. FIRST 5 ROWS (raw, uncleaned)")

sample = con.execute("SELECT * FROM raw_scrape").df()
sample[sample['source_url'].notnull()].head()


  3. FIRST 5 ROWS (raw, uncleaned)


,scrape_date,category,company_name,phone,email,website,employee_count,city,country,scraped_at,_source_file,_file_date,source_url
121717,2026-06-24,ERP,PrimeChainFlow,4349979873,info@primechainflow.com,https://www.primechainflow.com,297,Boston,US,2026-06-24T04:05:01Z,scrape_2026-06-24.csv,2026-06-24,https://primechainflow.com/profile
121718,2026-06-24,LEGAL,CedarClauseSolutions,02648853830,contact@cedarclausesolutions.co,https://cedarclausesolutions.co,11,Atlanta,US,2026-06-24T05:02:05Z,scrape_2026-06-24.csv,2026-06-24,https://cedarclausesolutions.co/profile
121719,2026-06-24,Legal,PrimeMatter Base,1-973-365-8286,info@primematterbase.com,https://www.primematterbase.com,5,Islamabad,PK,2026-06-24T02:48:06Z,scrape_2026-06-24.csv,2026-06-24,https://primematterbase.com/profile
121720,2026-06-24,HR,Iron CultureSolutions,+17594527727,contact@ironculturesolutions.io,https://www.ironculturesolutions.io,42,Munich,DE,2026-06-24T01:14:39Z,scrape_2026-06-24.csv,2026-06-24,https://ironculturesolutions.io/profile
121721,2026-06-24,HR,ClearPerform Base,1-209-807-2153,contact@clearperformbase.com,https://clearperformbase.com,497,Boston,US,2026-06-24T02:43:37Z,scrape_2026-06-24.csv,2026-06-24,https://clearperformbase.com/profile


In [14]:

# ─────────────────────────────────────────────
# 4. ROWS PER DAY — how many rows each day?
# ─────────────────────────────────────────────
divider("4. ROWS PER SCRAPE DATE")

daily = con.execute("""
    SELECT
        scrape_date,
        COUNT(*) as row_count
    FROM raw_scrape
    GROUP BY scrape_date
    ORDER BY scrape_date
""").df()
print(daily.to_string(index=False))



  4. ROWS PER SCRAPE DATE
scrape_date  row_count
 2026-06-01       4650
 2026-06-02       4719
 2026-06-03       4763
 2026-06-04       4807
 2026-06-05       4886
 2026-06-06       4939
 2026-06-07       5002
 2026-06-08       5089
 2026-06-09       5148
 2026-06-10       5211
 2026-06-11       5244
 2026-06-12       5410
 2026-06-13       5413
 2026-06-14       5500
 2026-06-15       5441
 2026-06-16       5592
 2026-06-17       4488
 2026-06-18       5748
 2026-06-19       5804
 2026-06-20       5890
 2026-06-21       6002
 2026-06-22       5935
 2026-06-23       6035
 2026-06-24       6157
 2026-06-25       6213
 2026-06-26       6184
 2026-06-27       6868
 2026-06-28       6387
 2026-06-29       6427
 2026-06-30       6499
 2026-07-01       6495
 2026-07-02       6583
 2026-07-03       6583
 2026-07-04       6572
 2026-07-05       6547
 2026-07-06       6511
 2026-07-07       6538
 2026-07-08       6577
 2026-07-09       6616
 2026-07-10       6554
 2026-07-11       6602
 2026-0

In [15]:

# ─────────────────────────────────────────────
# 5. CATEGORIES — what categories exist?
# ─────────────────────────────────────────────
divider("5. CATEGORIES (unique values + counts)")

cats = con.execute("""
    SELECT
        category,
        COUNT(*) as row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM raw_scrape
    GROUP BY category
    ORDER BY row_count DESC
""").df()
print(cats.to_string(index=False))



  5. CATEGORIES (unique values + counts)
    category  row_count  pct
         ERP      61485 25.1
          HR      48756 19.9
     Medical      48593 19.8
MSP Platform      39771 16.2
       Legal      24189  9.9
         LMS      13389  5.5
       legal       4035  1.6
      LEGAL        2954  1.2
         Lgl       1996  0.8
    category          1  0.0


In [16]:

# ─────────────────────────────────────────────
# 6. PHONE FORMATS — what does messy data look like?
# ─────────────────────────────────────────────
divider("6. PHONE NUMBER SAMPLES (first 15 unique formats)")

phones = con.execute("""
    SELECT DISTINCT phone
    FROM raw_scrape
    WHERE phone IS NOT NULL
    LIMIT 15
""").df()
for p in phones["phone"]:
    print(f"  {p}")



  6. PHONE NUMBER SAMPLES (first 15 unique formats)
  (585) 433-2803
  +17609691572
  (489) 926-1847
  339-329-7960 ext. 86
  9154003246
  4877943963 ext. 71
  (277) 529-5036
  412-745-4703
  2204574516
  (733) 961-9518
  +14965080891
  1-729-306-3484
  (867) 596-8084
  1-483-548-3758
  239-630-3186


In [17]:

# ─────────────────────────────────────────────
# 7. NULL COUNTS — where is data missing?
# ─────────────────────────────────────────────
divider("7. NULL / MISSING VALUES PER COLUMN")

cols = ["scrape_date", "category", "company_name", "phone",
        "email", "website", "employee_count", "city", "country", "source_url"]

print(f"  {'Column':<20} {'Nulls':>8} {'%':>8}  {'Visual'}")
print(f"  {'-'*20} {'-'*8} {'-'*8}  {'-'*20}")

for col in cols:
    result = con.execute(f"""
        SELECT COUNT(*) as nulls
        FROM raw_scrape
        WHERE {col} IS NULL OR TRIM({col}) = ''
    """).fetchone()[0]
    pct = result / total_rows * 100
    bar = "█" * int(pct / 5)  # 1 block per 5%
    print(f"  {col:<20} {result:>8,} {pct:>7.1f}%  {bar}")



  7. NULL / MISSING VALUES PER COLUMN
  Column                  Nulls        %  Visual
  -------------------- -------- --------  --------------------
  scrape_date                 0     0.0%  
  category                    0     0.0%  
  company_name                0     0.0%  
  phone                       0     0.0%  
  email                   7,348     3.0%  
  website                     0     0.0%  
  employee_count          5,679     2.3%  
  city                    3,604     1.5%  
  country                     0     0.0%  
  source_url            121,717    49.6%  █████████


In [18]:

# ─────────────────────────────────────────────
# 8. EMPLOYEE COUNT — what values appear?
# ─────────────────────────────────────────────
divider("8. EMPLOYEE COUNT SAMPLES (20 unique values)")

emp = con.execute("""
    SELECT DISTINCT employee_count
    FROM raw_scrape
    WHERE employee_count IS NOT NULL
    LIMIT 20
""").df()
print("  Values found:")
for e in emp["employee_count"]:
    print(f"    {e}")



  8. EMPLOYEE COUNT SAMPLES (20 unique values)
  Values found:
    553
    42
    347
    289
    80
    70
    144
    1202
    2839
    156
    324
    217
    286
    3766
    659
    2673
    647
    403
    205
    93


In [ ]:

# ─────────────────────────────────────────────
# 9. COUNTRIES — where are these companies?
# ─────────────────────────────────────────────
divider("9. TOP 10 COUNTRIES")

countries = con.execute("""
    SELECT
        country,
        COUNT(*) as count
    FROM raw_scrape
    WHERE country IS NOT NULL
    GROUP BY country
    ORDER BY count DESC
    LIMIT 10
""").df()
print(countries.to_string(index=False))


In [19]:

# ─────────────────────────────────────────────
# 10. SCHEMA CHANGE — when did source_url appear?
# ─────────────────────────────────────────────
divider("10. SCHEMA CHANGE — source_url presence by date")

schema_change = con.execute("""
    SELECT
        scrape_date,
        COUNT(*) as total_rows,
        COUNT(source_url) as rows_with_source_url
    FROM raw_scrape
    GROUP BY scrape_date
    ORDER BY scrape_date
""").df()
print(schema_change.to_string(index=False))

con.close()

print(f"\n{'='*60}")
print("  EXPLORATION COMPLETE")
print(f"{'='*60}\n")


  10. SCHEMA CHANGE — source_url presence by date
scrape_date  total_rows  rows_with_source_url
 2026-06-01        4650                     0
 2026-06-02        4719                     0
 2026-06-03        4763                     0
 2026-06-04        4807                     0
 2026-06-05        4886                     0
 2026-06-06        4939                     0
 2026-06-07        5002                     0
 2026-06-08        5089                     0
 2026-06-09        5148                     0
 2026-06-10        5211                     0
 2026-06-11        5244                     0
 2026-06-12        5410                     0
 2026-06-13        5413                     0
 2026-06-14        5500                     0
 2026-06-15        5441                     0
 2026-06-16        5592                     0
 2026-06-17        4488                     0
 2026-06-18        5748                     0
 2026-06-19        5804                     0
 2026-06-20        5890      